# edge-grader: stage-1 grading baseline on Kaggle

Runs the untouched Gemma 4 model (the **same QAT Q4_K_XL GGUF files as `agentic-edge`**) in `llama-server` with the lab's baseline flags, then grades SciEntsBank answers with `edge-grader-baseline`, or answers the lab's 300-question MMLU-Pro subset with `edge-grader-mmlupro` (`TASK = "mmlupro"`, stage 2), or runs the topic benchmarks: math with `edge-grader-gsm` (`TASK = "gsm"`: GSM8K and GSM-Symbolic) and post-cutoff news with `edge-grader-news` (`TASK = "news"`: LiveNewsBench).

**Before running** (notebook sidebar):
1. **Settings → Accelerator:** GPU T4 (x1 is enough) or P100.
2. **Settings → Internet:** on.
3. **Add-ons → Secrets:** add `GITHUB_TOKEN`, a fine-grained GitHub token with read-only *Contents* access to `gnotnek/edge-grader` only. Never paste it into a cell.

**What this measures:** accuracy only. Latency here is a T4 GPU and says nothing about the Pi or Jetson; board numbers come from `scripts/board_grade.sh`.

Results land in `/kaggle/working/edge-grader/results/kaggle/`. Download that folder from the notebook's Output tab, copy it into `results/kaggle/` in your clone, and commit it.

In [ ]:
# What to run.
TASK = "scientsbank"   # or "mmlupro": the lab's 3 x 100 MMLU-Pro questions, lm-eval's exact prompt
                       # or "gsm" (math) / "news" (LiveNewsBench): the topic benchmarks
MODELS = ["e4b", "e2b"]
SPLITS = ["test_ua"]   # scientsbank: test_ua 540, test_uq 733, test_ud 4562 answers
SUBSETS = ["s1", "s2", "s3"]   # mmlupro
GSM_SETS = ["tiny", "gsm8k", "symbolic"]   # gsm: tiny = the lab's tinyGSM8k replayed (runner check),
                       # gsm8k = 100 originals, symbolic = their GSM-Symbolic variants (10 each)
GSM_TINY_E4B_TOKENS = 256   # the lab ran E4B tinyGSM8k at 256 tokens only; E2B at 1,024 (the default)
N = 100          # scientsbank: answers per split; mmlupro: first N questions per subset;
                 # gsm: first N per set; news: first N of 200. 0 = all
# Start small: check unparsed (scientsbank) or no_answer / hit_token_cap (mmlupro), then N = 0.
# MMLU-Pro is chain-of-thought, up to 2,048 tokens per question: on a T4 budget roughly
# 1-1.5 h for E4B and 30-45 min for E2B per 300 questions, per condition.
# GSM: 1,200 questions per model, up to 1,024 tokens each. News: 200 questions, up to 2,048.

# Stage 2 (RAG): set RAG to a committed retrieval file to add course material to the prompt.
# None = the stage-1 baseline prompt, unchanged. The main condition is English:
#   RAG = "results/rag/retrieval-scientsbank-en.bge-small-en-k5.json"
# Optional, Indonesian material (docs/guides/rag.md):
#   RAG = "results/rag/retrieval-scientsbank-id.bge-m3-k5.json"
# MMLU-Pro (TASK = "mmlupro"), OpenStax material:
#   RAG = "results/rag/retrieval-mmlupro-en.bge-small-en-k5.json"
# News (TASK = "news"), the questions' own source articles, pooled:
#   RAG = "results/rag/retrieval-news-en.bge-small-en-k5.json"
# GSM (TASK = "gsm"), OpenStax Prealgebra 2e (phase A; set GSM_SETS = ["gsm8k", "symbolic"]):
#   RAG = "results/rag/retrieval-gsm-en.bge-small-en-k5.json"
RAG = None
# Phase B1: thinking on (agentic-edge's -rea on --reasoning-budget 320 --reasoning-format none).
# scientsbank only, no RAG. Files are named think-*.
THINKING = False
# Stage 3: a fine-tuned adapter per model, served with --lora on the same GGUF (any TASK).
# From the private dataset edge-grader-adapters, e.g.
#   LORA = {"e2b": "/kaggle/input/edge-grader-adapters/e2b-ft.gguf"}. Labels get -ft.
LORA = None
K = 3                          # passages per prompt (the retrieval file stores 5)
CHUNKS_INPUT = "/kaggle/input"  # with RAG: the private chunks dataset, searched recursively


In [ ]:
# Settings checked first, so a Save & Run All version fails in seconds, not after the build.
import os

assert TASK in ("scientsbank", "mmlupro", "gsm", "news"), f"TASK {TASK!r}: lowercase, one of the four"
if LORA:
    !find /kaggle/input -name "*.gguf" -exec ls -l {{}} \;
    for key in MODELS:
        assert os.path.exists(LORA[key]), f"LORA[{key!r}] not found: {LORA[key]} (dataset attached? path above)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Build llama-server with CUDA (about 10–15 min, once per session)

llama.cpp is pinned to commit `a3a1c47` (2026-10-05). Change it only on purpose, and re-run the baseline with any new build.


In [ ]:
%%bash
set -e
nvidia-smi -L || { echo "No GPU in this session: Settings -> Accelerator -> GPU, then restart the session"; exit 1; }
cd /tmp
# Pinned, so every run uses the same llama.cpp. Unpinned, Kaggle got whatever was newest that day:
# the stage-1 baseline ran on 86ea01d, the RAG-en runs on a3a1c47 (docs/results/rag.md).
LLAMA_CPP_COMMIT=a3a1c4747fdc0dcad40b3946108b89375d9a7d0e
if [ "$(git -C llama.cpp rev-parse HEAD 2>/dev/null)" != "$LLAMA_CPP_COMMIT" ]; then
  rm -rf llama.cpp && mkdir llama.cpp && cd llama.cpp
  git init -q && git remote add origin https://github.com/ggml-org/llama.cpp
  git fetch -q --depth 1 origin "$LLAMA_CPP_COMMIT" && git checkout -q FETCH_HEAD
  cd /tmp
fi
export PATH=/usr/local/cuda/bin:$PATH
CUDA_ROOT=$(dirname "$(dirname "$(readlink -f "$(command -v nvcc)")")")
# CMake can miss libcuda on Kaggle ("links to CUDA::cuda_driver but the target was not found"):
# the driver library sits outside the toolkit (e.g. /usr/local/nvidia/lib64). Find it and pass it in.
LIBCUDA=$( { ls "$CUDA_ROOT"/lib64/stubs/libcuda.so "$CUDA_ROOT"/targets/x86_64-linux/lib/stubs/libcuda.so 2>/dev/null;
             ldconfig -p | awk '/libcuda\.so/ {print $NF}';
             find /usr /opt -name 'libcuda.so*' 2>/dev/null; } | head -1 )
echo "nvcc: $(command -v nvcc)  CUDA_ROOT: $CUDA_ROOT  libcuda: ${LIBCUDA:-NOT FOUND}"
[ -n "$LIBCUDA" ] || { echo "GPU is on but libcuda was not found; paste this output to debug"; exit 1; }
rm -rf llama.cpp/build   # a failed configure caches the bad result
# 75 = T4, 60 = P100. Building only these keeps compile time down.
cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES="60;75" -DLLAMA_CURL=OFF \
  -DCUDAToolkit_ROOT="$CUDA_ROOT" -DCMAKE_CUDA_COMPILER="$CUDA_ROOT/bin/nvcc" \
  -DCUDA_cuda_driver_LIBRARY="$LIBCUDA" > /tmp/cmake.log 2>&1 || { tail -30 /tmp/cmake.log; exit 1; }
cmake --build llama.cpp/build -j"$(nproc)" --target llama-server > /tmp/build.log 2>&1 || { tail -30 /tmp/build.log; exit 1; }
cd llama.cpp && echo "llama.cpp $(git rev-parse --short HEAD)" && ls -la build/bin/llama-server

## 2. Download the lab's model files

In [ ]:
from huggingface_hub import hf_hub_download

FILES = {
    "e2b": ("unsloth/gemma-4-E2B-it-qat-GGUF", "gemma-4-E2B-it-qat-UD-Q4_K_XL.gguf"),
    "e4b": ("unsloth/gemma-4-E4B-it-qat-GGUF", "gemma-4-E4B-it-qat-UD-Q4_K_XL.gguf"),
}
MODEL_PATHS = {k: hf_hub_download(repo, fname, local_dir="/tmp/models") for k, (repo, fname) in FILES.items() if k in MODELS}
MODEL_PATHS

## 3. Get the code and install it with uv

In [ ]:
import base64
import os
import shutil
import subprocess

from kaggle_secrets import UserSecretsClient

REPO = "/kaggle/working/edge-grader"
# The token reaches git through the environment as an HTTP header, as actions/checkout does: never
# in the command line (a failed clone's error, shown in the notebook output, would print it) and
# never in .git/config.
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
env = {**os.environ, "GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
       "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"}
del token, auth
shutil.rmtree(REPO, ignore_errors=True)
clone = subprocess.run(["git", "clone", "-q", "https://github.com/gnotnek/edge-grader.git", REPO],
                       env=env, capture_output=True, text=True, check=False)
del env
if clone.returncode != 0:
    raise RuntimeError(f"git clone failed (check the GITHUB_TOKEN secret): {clone.stderr.strip()}")
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)

In [ ]:
!pip install -q uv
!cd /kaggle/working/edge-grader && uv sync -q && uv run pytest -q 2>&1 | tail -1

### RAG only: copy in the chunks
The retrieval file holds chunk ids only; the passage text comes from the chunks built on the Mac, uploaded as a **private** Kaggle dataset (see `notebooks/kaggle_rag_index.ipynb`). This checks them against the committed sha256. Skipped when `RAG = None`.

News chunks aren't uploaded: they're rebuilt here from LiveNewsBench's file (downloaded at a pinned commit, sha256 checked) and must match the committed sha256 too.

In [ ]:
import hashlib
import json
import re
import shutil
import subprocess
from pathlib import Path

if RAG:
    REPO = "/kaggle/working/edge-grader"
    bench, lang = re.search(r"(scientsbank|mmlupro|news|gsm)-(\w+)\.", RAG).groups()
    assert bench == TASK, f"RAG file is for {bench}, TASK is {TASK}"
    record = json.loads(Path(REPO, f"results/rag/chunks-{bench}.json").read_text())
    dest = Path(REPO, "data/rag/chunks")
    dest.mkdir(parents=True, exist_ok=True)
    for name, info in record["files"].items():
        if name != f"{bench}-{lang}.jsonl":
            continue  # only the language this run uses
        if bench == "news":  # rebuilt from the pinned LiveNewsBench file, not uploaded
            subprocess.run(["uv", "run", "python", "-m", "edge_grader.rag.corpus", "--benchmark", "news",
                            "--lang", lang], cwd=REPO, check=True)
            subprocess.run(["git", "checkout", "results/rag/chunks-news.json"], cwd=REPO, check=True)
        else:
            found = list(Path(CHUNKS_INPUT).rglob(name))
            if not found:
                raise FileNotFoundError(f"{name} not under {CHUNKS_INPUT}; add the chunks dataset as an input")
            shutil.copy(found[0], dest / name)
        got = hashlib.sha256((dest / name).read_bytes()).hexdigest()
        assert got == info["sha256"], f"{name}: sha256 differs from the committed record; re-upload the chunks"
        print(f"{name}: {info['chunks']} chunks, sha256 matches")
    # Summary files are named rag-<model>-kaggle-<gpu>-<lang>-k<K>-<split>-...
    RAG_ARGS, RAG_TAG = f"--rag {RAG} --k {K}", f"-{lang}-k{K}"
else:
    RAG_ARGS, RAG_TAG = "", ""
print("prompt:", f"RAG {RAG}, k={K}" if RAG else "stage-1 baseline")
RAG_TAG += "-ft" if LORA else ""  # stage 3: the label says the adapter was on


## 4. Start the server and grade

In [ ]:
import os
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

GPU = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     check=True, capture_output=True, text=True).stdout.split("\n")[0].split()[-1].lower()  # e.g. "t4"

def start_server(key):
    # agentic-edge baseline flags (thinking off unless THINKING), plus -ngl 99 to put the whole model on the GPU.
    cmd = ["/tmp/llama.cpp/build/bin/llama-server", "-m", MODEL_PATHS[key], "-c", "8192",
           "--host", "127.0.0.1", "--port", "8080", "-ngl", "99",
           *(["-rea", "on", "--reasoning-budget", "320", "--reasoning-format", "none"] if THINKING
             else ["-rea", "off", "--reasoning-budget", "-1"]),
           "--cache-ram", "0", *(["--lora", LORA[key]] if LORA else [])]
    with open(f"/tmp/server-{key}.log", "w") as log:
        # One GPU only (Kaggle may give two T4s): the model fits, and runs stay comparable.
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
    for _ in range(150):
        try:
            if urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2).status == 200:
                return proc
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            pass  # still loading
        if proc.poll() is not None:
            break
        time.sleep(2)
    proc.kill()
    # /tmp is gone after a Save & Run All version: the cause has to be in the output.
    print("\n".join(Path(f"/tmp/server-{key}.log").read_text().splitlines()[-40:]))
    raise RuntimeError(f"llama-server failed; see /tmp/server-{key}.log (last lines above)")

assert not (THINKING and (RAG or TASK != "scientsbank")), "thinking: scientsbank, no RAG"
THINK_ARGS = "--thinking" if THINKING else ""
for key in MODELS:
    server = start_server(key)
    try:
        if TASK == "mmlupro":
            print(f"== {key} MMLU-Pro {' '.join(SUBSETS)} ==")
            !cd /kaggle/working/edge-grader && uv run edge-grader-mmlupro --label {key}-kaggle-{GPU}{RAG_TAG} --subset {' '.join(SUBSETS)} --n {N} --out results/kaggle {RAG_ARGS}
        elif TASK == "gsm":
            assert not (RAG and "tiny" in GSM_SETS), "tiny is the runner check: no RAG"
            for gsm_set in GSM_SETS:
                tokens = GSM_TINY_E4B_TOKENS if (gsm_set == "tiny" and key == "e4b") else 1024
                tag = f"-{tokens}tok" if tokens != 1024 else ""
                print(f"== {key} GSM {gsm_set} ({tokens} tokens) ==")
                !cd /kaggle/working/edge-grader && uv run edge-grader-gsm --label {key}-kaggle-{GPU}{tag}{RAG_TAG} --set {gsm_set} --max-tokens {tokens} --n {N} --out results/kaggle {RAG_ARGS}
        elif TASK == "news":
            print(f"== {key} news ==")
            !cd /kaggle/working/edge-grader && uv run edge-grader-news --label {key}-kaggle-{GPU}{RAG_TAG} --n {N} --out results/kaggle {RAG_ARGS}
        else:
            for split in SPLITS:
                print(f"== {key} {split} ==")
                !cd /kaggle/working/edge-grader && uv run edge-grader-baseline --label {key}-kaggle-{GPU}{RAG_TAG} --split {split} --n {N} --out results/kaggle {RAG_ARGS} {THINK_ARGS}
    finally:
        server.terminate()
        server.wait()

## 5. Results

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for f in sorted(Path("/kaggle/working/edge-grader/results/kaggle").glob("*.json")):  # baseline-*, rag-*, mmlupro-*, gsm-*, news-*
    r = json.loads(f.read_text())
    if r.get("kind") == "judged":
        continue  # news judge grades: shown by notebooks/kaggle_news_judge.ipynb
    m = {k: v for k, v in r["metrics"].items() if k not in ("per_subject", "per_instance_accuracy")}
    rows.append({"benchmark": r.get("benchmark", "scientsbank"), "stage": r["stage"], "label": r["label"],
                 "split": r.get("split", r.get("subset", r.get("set"))), **m})
pd.DataFrame(rows)

**Check before scaling up:**
- `unparsed` should be close to 0. If it isn't, look at the `raw` field in `results/kaggle/raw/*.jsonl` to see what the model replied, then fix the prompt or parser.
- `errors` should be 0.
- MMLU-Pro: `no_answer` (no "the answer is (X)") and `hit_token_cap` (stopped at 2,048 tokens) count as wrong, as in lm-eval. A handful per 100 is normal; the lab's runs had about 9.

- GSM: `no_number` and `hit_token_cap` should be near 0 at 1,024 tokens. The `tiny` set is the runner check: the lab's E2B got 81/100 at 1,024 tokens, E4B 75/100 at 256.
- News: `answered` (an `<answer>` tag) should be close to `n`. Accuracy comes later, from the judge: `notebooks/kaggle_news_judge.ipynb`, once these runs are committed.

Then set `N = 0` and add `test_uq` / `test_ud` to `SPLITS`.

## 6. Download the results
Kaggle only downloads single files, so this zips `results/kaggle/` into one file. Click the link, or download `results-kaggle.zip` from the Output panel. Then, on your machine:
```bash
unzip -o ~/Downloads/results-kaggle.zip -d results/kaggle
git add results/kaggle && git commit -m "Kaggle baseline: ..." && git push
```

In [ ]:
import shutil

from IPython.display import FileLink

shutil.make_archive("/kaggle/working/results-kaggle", "zip", "/kaggle/working/edge-grader/results/kaggle")
FileLink("results-kaggle.zip")